In [0]:
# STEP 1: Import Functions
from pyspark.sql.functions import col, lower, trim, regexp_replace, to_timestamp

In [0]:
# STEP 2: Load Data from Bronze
df_bronze = spark.table("workspace.default.bronze_tickets")

In [0]:
# STEP 3: Data Quality & Casting (The core of Silver Layer)
# Convert strings to their proper types to enable fast math and date filters.
df_silver = df_bronze \
    .withColumn("ticket_id", col("ticket_id").cast("int")) \
    .withColumn("customer_name", trim(col("customer_name"))) \
    .withColumn("customer_age", col("customer_age").cast("int")) \
    .withColumn("issue_complexity_score", col("issue_complexity_score").cast("float")) \
    .withColumn("status", lower(col("status"))) \
    .withColumn("priority", lower(col("priority")))

In [0]:
# STEP 4: Handling Timestamps
# Crucial for time-series analysis in the Gold layer
# Auto Loader usually reads everything as string. Let's fix that.
df_silver = df_silver \
    .withColumn("created_at", to_timestamp(col("ticket_created_date"))) \
    .withColumn("resolved_at", to_timestamp(col("ticket_resolved_date")))

In [0]:
# STEP 5: Deduplication Logic
# In real-world ingestion, we might get the same ticket twice.
# I keep only the most recent or unique occurrence.
df_silver_clean = df_silver.dropDuplicates(["ticket_id"])

In [0]:
# STEP 6: Write to Silver Table (Delta Format)
# I drop '_rescued_data' if it's empty to keep the table clean for Analysts
df_final_silver = df_silver_clean.drop("_rescued_data")

(df_final_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_tickets"))

print("Successfully processed Silver Layer with proper types.")

Successfully processed Silver Layer with proper types.
